In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsstoragertp9.dfs.core.windows.net",
    "QhKofrsyaZYX9oH3BH56HhGubXsbCv/h9hrkjJMXJpDaOpPbYCnXRvh2cvGf8UYo1kyThYW5jaz9+AStplQf8g=="
)

In [0]:
# Drop existing silver tables to start fresh
spark.sql("DROP TABLE IF EXISTS ecommerce_silver.sales.orders")
spark.sql("DROP TABLE IF EXISTS ecommerce_silver.sales.order_items")
spark.sql("DROP TABLE IF EXISTS ecommerce_silver.sales.payments")
spark.sql("DROP TABLE IF EXISTS ecommerce_silver.sales.customers")
spark.sql("DROP TABLE IF EXISTS ecommerce_silver.sales.products")
spark.sql("DROP TABLE IF EXISTS ecommerce_silver.sales.sellers")

print("All old silver tables dropped!")

All old silver tables dropped!


In [0]:
from pyspark.sql.functions import col, to_timestamp, trim, when

spark.sql("CREATE CATALOG IF NOT EXISTS ecommerce_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS ecommerce_silver.sales")

orders_raw = spark.table("ecommerce_bronze.sales.orders_raw")

orders_clean = orders_raw \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
    .filter(col("order_id").isNotNull()) \
    .dropDuplicates(["order_id"])

orders_clean.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_silver.sales.orders")

print(f"Orders silver: {orders_clean.count()} rows")
display(orders_clean.limit(5))

Orders silver: 99441 rows


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
f373335aac9a659de916f7170b8bc07a,f06a94a401e52fb019c72f2e8bbf6a2f,shipped,2018-03-17T15:32:31Z,2018-03-17T15:48:40Z,2018-03-20T21:08:28Z,null,2018-04-13T00:00:00Z
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,2018-03-09T19:08:26Z,2018-03-13T21:24:28Z,2018-04-11T12:53:50Z,2018-04-04T00:00:00Z
cc66dee6fbc18bb79903c3a2cc14ff52,19d3b3a2d4756af17603e2c35c7c2815,delivered,2018-04-12T14:37:29Z,2018-04-12T15:15:27Z,2018-04-16T16:23:53Z,2018-04-20T17:28:56Z,2018-05-07T00:00:00Z
f44cb69655f8e4d13e7aae7cdd3d3c25,eab62436056c6ce3853a17dd6892951a,delivered,2018-07-13T22:22:57Z,2018-07-13T22:35:20Z,2018-07-24T19:07:00Z,2018-07-25T14:03:41Z,2018-07-31T00:00:00Z
edcc6b79e8394346ba3ba21b00b4055e,08aea10c40f606e52597486db2b56a81,delivered,2018-04-29T16:03:47Z,2018-04-29T16:15:25Z,2018-05-02T08:25:00Z,2018-05-11T23:12:12Z,2018-05-25T00:00:00Z


In [0]:
items_raw = spark.table("ecommerce_bronze.sales.order_items_raw")

items_clean = items_raw \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double")) \
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
    .filter(col("order_id").isNotNull()) \
    .filter(col("price") > 0)

items_clean.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_silver.sales.order_items")

print(f"Order items silver: {items_clean.count()} rows")
display(items_clean.limit(5))

Order items silver: 112650 rows


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
88498659c4bfb68808d6b49a57a6ed17,1,24ef70f010919148b022e66b05862778,a938325a4b357fd23a6a4d5bb126408e,2018-03-07T17:30:31Z,89.0,29.02
884989e29f137d8ce42518666158b8ce,1,fd76f22bf3648e0dd4295e593def7664,6d66611d7c44cc30ce351abc49a68421,2018-07-11T16:31:30Z,39.9,13.86
8849e8100a1269ca2a5606e5a0fb6c07,1,1cd35eaf33a642d4df4c9146837f5908,cac4c8e7b1ca6252d8f20b2fc1a2e4af,2017-07-11T17:24:16Z,49.99,16.6
8849f02bf66502f56ce8c5173bb954df,1,5ae2f57d379f89f995f9444cdb6d5c60,d594982fd877af63ace38ea1fca27c76,2018-08-08T03:45:33Z,27.19,18.29
884b1394fc8888e6a877df86eb19e74c,1,437c05a395e9e47f9762e677a7068ce7,f84fa566034f5e8e880a07ec624c56af,2018-03-25T22:08:28Z,47.65,7.39


In [0]:
payments_raw = spark.table("ecommerce_bronze.sales.payments_raw")

payments_clean = payments_raw \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .withColumn("payment_installments", col("payment_installments").cast("int")) \
    .withColumn("payment_sequential", col("payment_sequential").cast("int")) \
    .filter(col("order_id").isNotNull()) \
    .filter(col("payment_value") > 0)

payments_clean.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_silver.sales.payments")

print(f"Payments silver: {payments_clean.count()} rows")
display(payments_clean.limit(5))

Payments silver: 103877 rows


order_id,payment_sequential,payment_type,payment_installments,payment_value
e1e32786504bf329dc6a2c4d80579a31,1,credit_card,7,345.94
38cb813a638594505086f338bb5d4cae,1,credit_card,5,68.16
ddb7a8a2786d0021954b09fd87426bec,1,credit_card,1,19.39
a0f7769c320fd4c1900add6490eab094,1,credit_card,3,141.74
e19633c5f388df807021cded7372d3bc,1,boleto,1,68.84


In [0]:
customers_raw = spark.table("ecommerce_bronze.sales.customers_raw")

customers_clean = customers_raw \
    .withColumn("customer_state", trim(col("customer_state"))) \
    .withColumn("customer_city", trim(col("customer_city"))) \
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("int")) \
    .filter(col("customer_id").isNotNull()) \
    .dropDuplicates(["customer_id"])

customers_clean.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_silver.sales.customers")

print(f"Customers silver: {customers_clean.count()} rows")
display(customers_clean.limit(5))

Customers silver: 99441 rows


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
843ff05b30ce4f75b6170b39c78a8ee5,8718d37e3e19a80c134708dbe5815732,95595,cidreira,RS
971fc118b16e18e76c8ed6d0b1d8a67c,cecc19ff12c206e6368dd1e9c22a848d,18300,capao bonito,SP
de281e7aa3ddf4dfa36b4253ce263ed7,9bd37e786ab07a6d839e98b20df52c7b,9170,santo andre,SP
5bb2321bf6c692d7f92ec6d97d5842e4,f9e6d4a90f55c42ee1bd337bd09c852b,12327,jacarei,SP
c1785b084efcd3e71a83bfa29576dae6,1682c2c899bc883a9a2f86bc5eb43f12,91750,porto alegre,RS


In [0]:
products_raw = spark.table("ecommerce_bronze.sales.products_raw")
category_raw = spark.table("ecommerce_bronze.sales.category_raw")

products_clean = products_raw \
    .join(category_raw, "product_category_name", "left") \
    .withColumn("product_weight_g", col("product_weight_g").cast("double")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("double")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("double")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("double")) \
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
    .filter(col("product_id").isNotNull()) \
    .dropDuplicates(["product_id"])

products_clean.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_silver.sales.products")

print(f"Products silver: {products_clean.count()} rows")
display(products_clean.limit(5))

Products silver: 32951 rows


product_category_name,product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
cama_mesa_banho,e4bb5bd18b0b3b6e9b7107ecf4fd11cc,59,255,1,1800.0,45.0,15.0,35.0,bed_bath_table
casa_construcao,ed464125465d8ab04c3615963d27395c,56,904,6,700.0,39.0,7.0,39.0,home_construction
esporte_lazer,151259fe8ced305ca05dc771fc72d711,32,839,1,418.0,18.0,13.0,14.0,sports_leisure
telefonia,b3547583f59191a94cb364defe30e9ec,29,156,1,50.0,18.0,18.0,18.0,telephony
beleza_saude,e12f98550d5a1a612066b387f97c1970,59,1860,3,275.0,23.0,11.0,13.0,health_beauty


In [0]:
sellers_raw = spark.table("ecommerce_bronze.sales.sellers_raw")

sellers_clean = sellers_raw \
    .withColumn("seller_state", trim(col("seller_state"))) \
    .withColumn("seller_city", trim(col("seller_city"))) \
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("int")) \
    .filter(col("seller_id").isNotNull()) \
    .dropDuplicates(["seller_id"])

sellers_clean.write.mode("overwrite").format("delta") \
    .saveAsTable("ecommerce_silver.sales.sellers")

print(f"Sellers silver: {sellers_clean.count()} rows")
display(sellers_clean.limit(5))

Sellers silver: 3095 rows


seller_id,seller_zip_code_prefix,seller_city,seller_state
791cfcfe22fe4a771ece27f90017da92,14010,ribeirao preto,SP
e63e8bfa530fb16910dd6956e592bb81,7160,guarulhos,SP
4d600e08ecbe08258c79e536c5a42fee,85988,entre rios do oeste,PR
0b64bcdb0784abc139af04077d49a20e,92420,canoas,RS
c522be04e020c1e7b79f3acff36513d5,6501,santana de parnaiba,SP


In [0]:
tables = ["orders", "order_items", "payments", "customers", "products", "sellers"]

print("=== Silver Layer Summary ===")
for t in tables:
    count = spark.table(f"ecommerce_silver.sales.{t}").count()
    print(f"ecommerce_silver.sales.{t}: {count} rows")

=== Silver Layer Summary ===
ecommerce_silver.sales.orders: 99441 rows
ecommerce_silver.sales.order_items: 112650 rows
ecommerce_silver.sales.payments: 103877 rows
ecommerce_silver.sales.customers: 99441 rows
ecommerce_silver.sales.products: 32951 rows
ecommerce_silver.sales.sellers: 3095 rows
